## Process - phase 2, after qf-batch
- Load the cleaned beneficiary rows left by `clean_msa_1_before_qf_batch.ipynb`
- Read qf-batch's quotient familial verdict, join it back onto every child of a household
- Split the QF / AAH / AEEH eligibility routes
- Output to CSV

The eligibility windows are the shared ones in `../partners_lib.py`, the same the CNAF
runs on: QF 6-17 ans + quotient < 700, AAH 16-30 ans, AEEH 6-19 ans.

## Prerequisite
qf-batch.ts must have finished and written `MSA_QF_BATCH_OUTPUT_PATHFILE_2026`.

In [ ]:
import csv
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

# Every step of this phase is shared with the other partners routed through qf-batch, so
# clean_msa_lib is only needed for the columns MSA drops on top of the shared list.
import partners_lib as partners
import clean_msa_lib as msa

load_dotenv()

base_output_filepath = os.environ['DB_MSA_EXPORT_2026']
qf_batch_output_filepath = os.environ['MSA_QF_BATCH_OUTPUT_PATHFILE_2026']

# Written by clean_msa_1_before_qf_batch.ipynb - keep both notebooks on the same value.
# qf-batch-workdir sits in the partners/ folder rather than this one: every partner routed
# through qf-batch shares it, so the files inside stay partner-prefixed.
msa_intermediate_filepath = os.environ.get(
    'MSA_INTERMEDIATE_PATHFILE_2026',
    str(Path.cwd().parent / 'qf-batch-workdir' / 'msa_2026_pre_qf_batch.parquet'))

In [ ]:
# Cleaned, deduplicated beneficiary rows left by phase 1. They still carry the qf-batch
# pivot columns, dropped below once the routes have been computed from them.
df_valid_no_duplicate = pd.read_parquet(msa_intermediate_filepath)

print(f"{len(df_valid_no_duplicate)} beneficiary row(s) read from {msa_intermediate_filepath}")

In [ ]:
# qf-batch.ts runs out-of-band (can take up to a week) - read its verdict back in here.
# It records the raw quotient (qf_value), not an eligibility boolean: the threshold is
# applied below, in the QF route cell.
df_qf_batch_output = pd.read_csv(qf_batch_output_filepath, dtype=str, keep_default_na=False)
qf_value_by_allocataire = partners.build_qf_value_lookup(df_qf_batch_output)

# One quotient_familial value per household, fanned out to every child row of that
# allocataire. Rows without a value (404, error, non-ARS) stay NaN.
df_routes = partners.attach_qf_value(df_valid_no_duplicate, qf_value_by_allocataire)

In [ ]:
## drop the columns now folded into the JSON columns, plus the qf-batch pivot-only ones
## (partners.FINAL_COLUMNS_TO_DROP, extended with what only MSA carries - see
## msa.MSA_EXTRA_COLUMNS_TO_DROP). Dropped from df_final only: df_routes keeps them, and
## its qf_value column, so neither leaks into the exported CSV.
df_final = partners.drop_intermediate_columns(
    df_valid_no_duplicate, partners.FINAL_COLUMNS_TO_DROP + msa.MSA_EXTRA_COLUMNS_TO_DROP)

In [ ]:
# Quotient familial route: 6-17 ans révolus, household quotient must also clear the threshold
# (partners.QF_MAX). The same window already trimmed the qf-batch input in phase 1; re-applied
# here per beneficiary because an eligible household can also hold out-of-window children.
# The routes are computed on df_routes, which alone carries qf_value, and select the
# matching rows of df_final.
df_final_jeune = partners.select_eligible_by_index(df_final, partners.qf_eligible_index(df_routes))

In [ ]:
# AAH route: 16-30 ans révolus - situation already settled from MSA's prestation
df_final_aah = partners.select_eligible_by_index(df_final, partners.aah_eligible_index(df_routes))

In [ ]:
# AEEH route: 6-19 ans révolus - no quotient_familial call needed, MSA already flags it
# (it spells the prestation "AEH")
df_final_aeeh = partners.select_eligible_by_index(df_final, partners.aeeh_eligible_index(df_routes))

In [ ]:
# Merge QF, AAH and AEEH routes
df_final_jeune_and_aah = pd.concat(
    [df_final_jeune, df_final_aah, df_final_aeeh], ignore_index=True).reset_index(drop=True)

In [ ]:
# Cast to string
df_final_jeune_and_aah.loc[:,'date_naissance'] = df_final_jeune_and_aah['date_naissance'].astype(str)

In [ ]:
# output to CSV files
df_final_jeune_and_aah.to_csv(base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

In [ ]:
print(f"{len(df_final_jeune)} df_final_jeune")
print(f"{len(df_final_aah)} df_final_aah")
print(f"{len(df_final_aeeh)} df_final_aeeh")
print(f"{len(df_final_jeune_and_aah)} jeune, aah and aeeh")